# Week 1 — Introduction to Julia for Economists

**Programming & Numerical Methods for Economics** · The University of Edinburgh · Juan Zurita

**Learning goals.** By the end of this notebook you can:

1. Use variables, vectors, and matrices in Julia
2. Write functions and loops
3. Plot economic data
4. Build your first economic simulation (the Solow growth model)

> **Running this notebook:** in Google Colab, set *Runtime → Change runtime type → Julia*. Locally, install [Julia](https://julialang.org/downloads/) and IJulia.

## 1. Variables and types

Julia is fast like C but reads like maths. Assign variables with `=`; Julia infers the type.

In [ ]:
# Economic variables
gdp = 2.85e12        # UK GDP in dollars (Float64)
population = 68_350_000   # underscores make big numbers readable (Int64)
country = "United Kingdom"

gdp_per_capita = gdp / population
println("GDP per capita: ", round(gdp_per_capita, digits=2))

In [ ]:
# Julia supports Unicode — write economics like economics
α = 0.3          # capital share  (type \alpha then press TAB)
β = 0.96         # discount factor
println("α + β = ", α + β)

**Exercise 1.1** — Create variables for an economy: consumption `C = 1.8e12`, investment `I = 0.5e12`, government spending `G = 0.6e12`, and net exports `NX = -0.05e12`. Compute GDP with the expenditure identity and print it.

In [ ]:
# Your answer here


## 2. Vectors and matrices

Economic data lives in arrays: time series are vectors, panel data are matrices.

In [ ]:
# A time series of annual growth rates (percent)
growth = [2.1, 1.7, -9.3, 7.6, 4.1, 0.3, 0.9]
years  = 2018:2024      # a range

println("Mean growth: ", sum(growth)/length(growth))
println("Worst year: ", years[argmin(growth)])

In [ ]:
# Matrices: an input-output style example
A = [0.2 0.3;
     0.1 0.4]
x = [100.0, 50.0]

A * x        # matrix-vector product — no special syntax needed

In [ ]:
# Broadcasting: apply an operation element-wise with a dot
real_growth = growth .- 2.0      # subtract 2% inflation from every entry
positive_years = growth .> 0     # element-wise comparison
println(real_growth)
println(positive_years)

**Exercise 2.1** — Using `growth` above, compute the *cumulative* level of GDP relative to 2017 (=100). Hint: `cumprod(1 .+ growth ./ 100)`.

In [ ]:
# Your answer here


## 3. Functions and control flow

Functions are the building blocks of every model you will write in this course.

In [ ]:
# Cobb-Douglas production function — three equivalent definitions
function production(K, L, α)
    return K^α * L^(1-α)
end

produce(K, L; α=0.3) = K^α * L^(1-α)     # one-liner with a keyword argument

Y = production(100.0, 50.0, 0.3)
println("Output: ", round(Y, digits=2))

In [ ]:
# Control flow: a simple progressive tax
function tax(income)
    if income <= 12_570
        return 0.0
    elseif income <= 50_270
        return 0.20 * (income - 12_570)
    else
        return 0.20 * (50_270 - 12_570) + 0.40 * (income - 50_270)
    end
end

for inc in [10_000, 30_000, 80_000]
    println("Income £", inc, " → tax £", round(tax(inc), digits=0))
end

**Exercise 3.1** — Write a function `present_value(payment, r, T)` that returns the present value of receiving `payment` every year for `T` years at interest rate `r`. Check: `present_value(100, 0.05, 10) ≈ 772.17`.

In [ ]:
# Your answer here


## 4. Plotting

We use `Plots.jl`. The first plot takes a moment to compile — that is normal.

In [ ]:
using Plots

plot(years, growth,
     marker = :circle,
     linewidth = 2,
     label = "GDP growth",
     xlabel = "Year",
     ylabel = "Percent",
     title = "Annual GDP growth")
hline!([0], linestyle = :dash, color = :gray, label = "")

## 5. Application: the Solow growth model

Everything together. Capital evolves as

$$k_{t+1} = s \, k_t^{\alpha} + (1 - \delta) k_t$$

where $s$ is the saving rate, $\alpha$ the capital share, and $\delta$ depreciation. We simulate the path of capital from a low starting point and watch convergence to the steady state.

In [ ]:
function solow_path(; s=0.25, α=0.3, δ=0.05, k0=1.0, T=100)
    k = zeros(T)
    k[1] = k0
    for t in 1:T-1
        k[t+1] = s * k[t]^α + (1 - δ) * k[t]
    end
    return k
end

k = solow_path()
k_star = (0.25/0.05)^(1/(1-0.3))    # analytical steady state: (s/δ)^(1/(1-α))

plot(k, linewidth=2, label="capital path", xlabel="t", ylabel="k")
hline!([k_star], linestyle=:dash, label="steady state")

**Exercise 5.1** — Simulate two economies, one with `s = 0.20` and one with `s = 0.30`, and plot both capital paths on the same figure. Which converges to a higher steady state, and does it converge faster or slower?

**Exercise 5.2 (harder)** — Extend `solow_path` to also return the path of *output* $y_t = k_t^\alpha$ and *consumption* $c_t = (1-s) y_t$. Plot consumption for `s ∈ [0.1, 0.2, ..., 0.9]` at $t = 100$. Which saving rate maximises long-run consumption? (You have just discovered the *golden rule*.)

In [ ]:
# Your answers here


---

## Summary

You can now write Julia code with variables, arrays, functions, loops, and plots — and you have simulated your first growth model. Next week: performance, linear algebra, and why Julia loops are fast.

**Further reading:** [QuantEcon Julia lectures 1–3](https://julia.quantecon.org/) · [Julia documentation](https://docs.julialang.org/)